# Migrate Your Own Machine Learning Pytorch Model for Inference on A SageMaker AI Inference Endpoint

### Summary
This notebook will showcase how to deploy your own machine learning model on a Amazon SageMaker AI Inference Endpoint using SageMaker AI's direct API endpoints using exclusively `boto3` (as opposed to using the `sagemaker` module). It uses a generic ResNet50 machine vision model as an example model to migrate. It showcases how to generate an inference script for your model (that will run on SageMaker AI's managed infastructure), register that model in SageMaker AI Model Registry, itemize that model into a SageMaker AI Model, create a SageMaker AI Endpoint Configuration and lastly deploy your model using these resources into a SageMaker AI Inference Endpoint that we will then invoke.

### Prerequisites
The following Notebook was developed and tested on a ml.g4dn.xlarge, Using a Pytorch 3.10 & the "PyTorch 2.0 GPU Optimized" Image via SageMaker Studio Classic. It also assumes the executing IAM role of this notebook has access to CloudWatch logs to showcase logging of SageMaker Inference Endpoints.

## Let's begin!
First we'll need to have some model we'd like to actually run on SageMaker. For this demo, we'll use a pre-trained machine vision model that's provided via PyTorch called ResNet50, so let's create it and use it! 

In [ ]:
from torchvision.io import read_image
from torchvision.models import resnet50, ResNet50_Weights
import torch
import time

# For this demo, let's assume we've trained a Resnet50 model. 
# (in reality we'll just use a pretrained model that comes with torchvision)
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights = weights).cuda()
model_name = f"resnet50-{int(time.time())}"
print(f"Our model name for this example will be {model_name}")

Let's first understand how our model runs inference. The script below firstly optimizes our model for inference, and then showcases how to features from a given image, feed it through our model and interpert the model's output.

In [ ]:
# Let's prepare our model for inference!
model.eval() # Put model into inference mode
compiled_model = torch.jit.script(model) # Compile into torchscript for optimized inference

# Let's run a brief inference with this model to verify everything's running fine
preprocess = weights.transforms()


def infer(image_fname):
    # Load the image...
    img = read_image(image_fname)

    # Preprocess the Image
    batch = preprocess(img).unsqueeze(0).cuda()

    # Run Inference
    prediction = compiled_model(batch).squeeze(0).softmax(0)

    # Translate output
    class_id = prediction.argmax().item()
    score = prediction[class_id].item()
    category_name = weights.meta["categories"][class_id]
    print(f"{category_name}")

infer("penguin.jpg")


Great! Our model works and is optimized for inference.

To migrate it into the AWS cloud, we'll have to save our compiled inference model so we can upload it to the AWS cloud later. Really what we want here are file(s) needed to load our model programatically on our host, which in this case would be a SageMaker AI Endpoint. Since we're using Torchscript to load our model programatically, we only need a `.pt` file.

In [ ]:
torch.jit.save(compiled_model, "resnet50-compiled-model.pt")

Next we will have to register our model in a SageMaker AI Model Registry. To register a model, we need to provide: 

1. Our model formatted as a **Model Artifact**
2. a **Deep Learning Container Image** which the model will run in

# Creating a Model Artifact
A Model Artifact is a compressed file (`.tar.gzip`) that will provide all files we need available on our host (so our SageMaker AI Inference Endpoint) to be able to programatically instantiate and invoke our machine learning model.

Typically, a Model Artifact will contain...

<ol>
    <li>Our our machine learning model (model's weights and biases serialized to a file via ML framework)</li>
    <li>An inference handler script (script with function hooks that will ran when the SageMaker Endpoint is invoked)</li>
    <li>Any auxiliary python modules that we might need inside out inference handler</li>
</ol>

The format of a Model Artifact for our project should look like the following:

## Model Artifact - Inference Script

In the beginning of this notebook we've compiled and saved our model as `resnet50-compiled-model.pt`, and since we won't need any auxiliary python modules installed (which would've been listed in `requirements.txt`), the only piece we have to fill next is the inference handler script.

An inference handler script is a python script that MUST contain these 4 functions:
1. `model_fn` -- Loads the model from the given Model Artifact files and configure it appropriatly inside the container
2. `input_fn` -- Extract all features from an inference request by converting the input data into a format that's digestable by our loaded model
3. `predict_fn` -- Run the extracted features/formatted data (that's been extracted/formatted by `input_fn`) through the machine learning model (that's been loaded in `model_fn`)
4. `output_fn` -- Convert the raw output from our model that was yeilded from `predict_fn` into the requested inference format

`model_fn` will be called upon initialization of the endpoint, and the other 3 will be called in sequence every time our SageMaker AI Inference Endpoint is invoked. 

To illustrate how logging can be done through our inference model as well, we've created a logger that outputs its data to `stdout`. By default, <a href='https://docs.aws.amazon.com/sagemaker/latest/dg/logging-cloudwatch.html'>all `stdout` messages will get written into a CloudWatch LogStream that SageMaker AI will automatically will create and manage for us</a>. We will showcase how to view these logs later on.

Below is what our inference handler script should look like:

In [ ]:
%%writefile inference.py
# inference.py

import sys
import os
import io
import logging

import numpy as np
import torch
import PIL.Image as Image
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights


# Create a logger
l = logging.Logger('resnet50_logger')

# Redirect logging output to stdout
handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter('[%(asctime)s] (%(levelname)s) %(name)s: %(message)s')
handler.setFormatter(formatter)
l.addHandler(handler)

l.info("Inference script loaded!")


# Feature extraction objects
weights = ResNet50_Weights.DEFAULT
preprocess = weights.transforms()

# Function that will load our model into memory from the provided model file
def model_fn(model_dir):
    l.info("Loading model!")
    l.info(model_dir)
    model_fname = os.path.join("model-artifact", "resnet50-compiled-model.pt")
    l.info(f"Model fname : {model_fname}")
    
    model = torch.jit.load(model_fname, map_location=torch.device("cuda"))
    l.info("Loading model done!")
    
    return model

# Function that will process the raw input of the invocation (in this case -- bytearray of a jpg image)
# into a format that our model will understand (in this case -- a tensor)
def input_fn(request_body, request_content_type):
    
    l.info("Entering input_fn")
    
    if request_content_type == "image/jpg":
        image_as_pil = Image.open(io.BytesIO(request_body))
        image_as_tensor = transforms.PILToTensor()(image_as_pil)
        batch = preprocess(image_as_tensor).unsqueeze(0).cuda()
        return batch
    else:
        raise ValueError(f"Model only accepts request of type 'image/jpg'")

# Function that will push the output of input_fn (tensor model input) through our machine learning model
def predict_fn(input_data, model):
    l.info("Entering predict_fn")
    prediction = model(input_data).squeeze(0).softmax(0)
    return prediction
        
# Function that will convert the output of the machine learning model (in this case -- a tensor)
# into the format that was requested when invoked (in this case -- plaintext)
def output_fn(prediction, response_content_type, context):
    l.info("Entering output_fn")
    
    if response_content_type == "text/plain":    
        class_id = prediction.argmax().item()
        score = prediction[class_id].item()
        category_name = weights.meta["categories"][class_id]
        return str(category_name)
    else:
        raise ValueError(f"Model only provides responses of type 'text/plain'")


Now that we have our inference handler script, let's create the Model Artifact tarball itself. 

When deploying our Inference Endpoint later this notebook, it will automatically load our Model Artifact from Amazon S3. So to prepare the model for Inference, we will need to upload our Model Artifact to an Amazon S3 bucket.

In [ ]:
import tarfile
import os
import os.path
import shutil

# Create the model artifact
os.mkdir("model-artifact")
torch.jit.save(compiled_model, "model-artifact/resnet50-compiled-model.pt")

os.mkdir("model-artifact/code")
shutil.copy("inference.py", "model-artifact/code/inference.py")
open("model-artifact/code/requirements.txt", "w").close()

os.mkdir("code")
shutil.copy("inference.py", "code/inference.py")

# Compress model artifact
model_artifact_fname = 'model.tar.gz'
with tarfile.open(model_artifact_fname, "w:gz") as tar:
    tar.add('code')
    tar.add('model-artifact')


In [ ]:
import boto3
import sagemaker
from botocore.exceptions import ClientError

sagemaker_session = sagemaker.Session()
s3_client = boto3.client("s3")
sagemaker_client = boto3.client(service_name="sagemaker")

# Fetch default bucket of current SageMaker session
bucket = sagemaker_session.default_bucket()

# Upload model artifact that we've created to S3 bucket
s3_client.upload_file(model_artifact_fname, bucket, f'{model_name}/{model_artifact_fname}')

# Variable for storing address of our model artifact in S3
model_artifacts_url = f"s3://{bucket}/{model_name}/{model_artifact_fname}"
print(f"Uploaded Model Artifact ({model_artifact_fname}) to {model_artifacts_url}")

## Model Artifact - Deep Learning Container
SageMaker AI deploys your machine learning model inside of a AWS Deep Learning Container that's hosted and maintained by AWS. [AWS Deep Learning Containers (DLCs)](https://github.com/aws/deep-learning-containers) are a set of Docker images for training and serving models. These containers manage running our model in memory, providing the right libraries/modules for our model, binding a network endpoint to serve our model and more. 

We provide many Deep Learning Containers catred to various ML framework (eg. PyTorch, Tensorflow and HuggingFace, etc), hardware architectures (x86, ARM, CUDA-based GPUs or Accelerators), version of python, etc. You are also able to build your own container image if needed for your model.

For this demo we'll select an appropriate Deep Learning Container from our Available containers, where we provide guidelines for which container will suit your model. If you're unsure, the `sagemaker` module has built-in functionality to automatically find the right Deep Learning Container for a given model by correctly Model instance.


In [ ]:

boto_session = boto3.session.Session()
region = boto_session.region_name
role = sagemaker.get_execution_role()
instance_type = "ml.g4dn.xlarge"

# OPTIONAL: Example code if how to automatically find your Deep Leanring Container if you're unsure which one to use
"""
from sagemaker.pytorch import PyTorchModel

sagemaker_model = PyTorchModel(
    model_data=model_artifacts_url,
    entry_point="inference.py",
    role=role,
    framework_version="2.0",
    py_version="py310",
    sagemaker_session=sagemaker_session
)
image_uri = sagemaker_model.serving_image_uri(instance_type=instance_type, region_name=region)
"""

image_uri = '763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-inference:2.0-gpu-py310'
print("Using the foollowing Deep Learning Container Image:")
print(image_uri)


Now that we have our Model Artifact and our Deep Learning Container, we're ready to register our model!

## Model Registry

SageMaker Model Registry can be accessed inside SageMaker Studio as a single place of governance for all machine learning models that are used during the lifecycle of a project. It enables users to orchestrate common workloads that are typical during the MLOps lifecylce (eg. deployment, model performance, validation/test performance, Inference Recommender jobs, etc)

To start, we'll need to instantiate a Model Package Group, which is a catalog of different versions of your ML Model as it improves over time. "Model Package" here is an itemization of the Model Artifact and metadata (eg. Container Image, Perferable Instace Type, etc) that are required to deploy your model. Over time as your train new versions of your model, you will have new versions you'd register in your "Group" of versioned models.

In [ ]:
# Create a Model Package Group: https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-model-group.html
import time
from time import gmtime, strftime

response = sagemaker_client.create_model_package_group(
    ModelPackageGroupName = model_name,
    ModelPackageGroupDescription = "Model package group for our resnet50 models"
)
                                                
model_package_group_arn = response["ModelPackageGroupArn"]
print(f"ModelPackageGroup ARN: {model_package_group_arn}")

Now that we've created our Model Package Group, we can register our current working model as a Model Package into our Model Package Group.

In [ ]:
inference_specs = { "Containers": [ {
                        "Image": image_uri,
                        "Environment": {
                            "SAGEMAKER_PROGRAM": "inference.py",
                            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
                            "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
                            "SAGEMAKER_REGION": region
                        },
                        "ModelDataUrl": model_artifacts_url,
                        "Framework": "PYTORCH",
                        "FrameworkVersion": "2.0"
                    }],
                    "SupportedContentTypes": ["image/jpg"],
                    "SupportedResponseMIMETypes": ["text/plain"]
}

r = sagemaker_client.create_model_package(ModelPackageGroupName=model_package_group_arn,
                                          ModelPackageDescription='Our resnet50 model package',
                                          InferenceSpecification=inference_specs,
                                          CertifyForMarketplace=False,
                                          ModelApprovalStatus="PendingManualApproval")

model_package_arn = r["ModelPackageArn"]

print(f"Model Package {model_package_arn} created successfuly in Model Registry!")

In [ ]:
model_artifacts_url

The benefit of having all our models stored in a centeral registry is that it enables us to review and compare our models before deploying them to production. It would be wise to review a newly trained machine learning model before deploying it to production (in the same manner one would have a code review session to overlook new code!), however for this demo we'll assume it's good to go. To approve our model into produciton, we'll set its status as 'approved'.

Also, note that in the latter part of our model package's ARN, its last two values are the model name itself, and then version of the model in that specific package.

(eg. arn:aws:sagemaker:REGION:ACCOUNT_ID:model-package/MODEL_PACKAGE/VERSION)


In [ ]:
model_package_update_input_dict = {
    "ModelPackageArn" : model_package_arn,
    "ModelApprovalStatus" : "Approved"
}

response = sagemaker_client.update_model_package(**model_package_update_input_dict)
print(f"Model package {model_package_arn} successfuly approved!")

Now that we've approved our model for production, lets deploy it.

## Deploy the model

Model Packages enable us to deploy our model onto a SageMaker AI Inference Endpoint. Again, this endpoint will run our model by running the provided Deep Learning Container and binding to that container our Model Artifact.

In [ ]:
# NOTE: sagemaker_model_name here is an identifier for our SageMaker Model (capital I and capital M!) 
# a SageMaker Model is a different resource in our account than the model versions we register in our Model Package,
# and we need to create a SageMaker Model from our Model Package in our model registry first, and then reference it
# in our Endpoint Configuration.

# So first we create the SageMaker Model...
sagemaker_model_name = f"{model_name}-sm-model"
r = sagemaker_client.create_model(ModelName=sagemaker_model_name,
                              ExecutionRoleArn=role,
                              Containers=[{"ModelPackageName": model_package_arn}])
print("Created SageMaker Model successfuly!")



# And then we create a Endpoint Config which references that SageMaker Model...
endpoint_name = f"{model_name}-endpoint"
endpoint_config_name = f"{endpoint_name}-config"
r = sagemaker_client.create_endpoint_config(EndpointConfigName=endpoint_config_name,
                                        ProductionVariants=[ {"ModelName": sagemaker_model_name,
                                                              "VariantName": "AllTraffic",
                                                              "InitialVariantWeight": 1,
                                                              "InitialInstanceCount": 1,
                                                              "InstanceType": instance_type}])
print("Created Endpoint Config successfuly!")


# And lastly we create that SageMaker Inference Endpoint using the Endpoint Config we've created!
r = sagemaker_client.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
print("Created SageMaker Inference Endpoint Successfuly!")

In [ ]:
import time

while True:
    r = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
    model_deployed = (r["EndpointStatus"] == "InService")
    if model_deployed:
        break
    print(f"SageMaker Inference Endpoint status is '{r['EndpointStatus']}'... Sleeping for 60 seconds and will check again!")
    time.sleep(60)
    
print("SageMaker Inference Endpoint is deployed and in service!") 

And we can now invoke it for inference! a SageMaker Inference Endpoint can be referenced either by it's assigned URL (that you can restrict/make available depending your needs) or by using our `boto3` and referncing the endpoint's name directly. For this notebook we will do the latter.

In [ ]:
def loadImage(fname):
    img_fname = os.path.basename(fname)
    img = open(img_fname, "rb")
    imgage_byte_arr = img.read()
    return imgage_byte_arr

sagemaker_runtime_client = boto3.client("sagemaker-runtime")
image_payload = loadImage("wolf.jpg")

# https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_runtime_InvokeEndpoint.html
response = sagemaker_runtime_client.invoke_endpoint(EndpointName=endpoint_name, 
                                                    Body=image_payload,
                                                    ContentType="image/jpg",
                                                    Accept="text/plain")

label = response["Body"].read()

label

Great! Our model works. To look into the logs we've defined in our inference handler, we can again use boto3 to programatically fetch our logs from CloudWatch. (NOTE: Your assigned SageMaker IAM role might not have permission to access CloudWatch logs. To fix this you need to provision the right CloudWatch access policy to the role, which you can do on IAM through the AWS Console)

In [ ]:
import boto3
import time

print(f"NOTE! The following code will only work if you allow the IAM role {role} access to CloudWatch's events.")

try:
    cloudwatch_logs_group = f"/aws/sagemaker/Endpoints/{endpoint_name}"
    cloudwatch_client = boto3.client("logs")


    # https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/logs/client/filter_log_events.html
    response = cloudwatch_client.filter_log_events(
        logGroupName=cloudwatch_logs_group,
        filterPattern="resnet50",
    )
    
    for e in response["events"]:
        print(e["message"])
        
except Exception as e:
    print(e)

In [ ]:
endpoint_name

## Clean Up (optional)


In [ ]:
shutil.rmtree("model-artifact")
shutil.rmtree("code")
os.remove("inference.py")
os.remove("model.tar.gz")
os.remove("resnet50-compiled-model.pt")


In [ ]:
# Clean up S3 Bucket
s3_client.delete_object(Bucket=bucket, Key=model_artifact_fname)

# Clean up model registry
sagemaker_client.delete_model_package(ModelPackageName=model_package_arn)
sagemaker_client.delete_model_package_group(ModelPackageGroupName=model_name)

# Delete SageMaker Model
sagemaker_client.delete_model(ModelName=sagemaker_model_name)

# Delete Model Endpoint Config
sagemaker_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)

# Delete Model Endpoint
sagemaker_client.delete_endpoint(EndpointName=endpoint_name)

# Clean up CloudWatch Logs Group
cloudwatch_client.delete_log_group(logGroupName=cloudwatch_logs_group)

print("Successfuly cleaned up AWS account of all resources created for this demo.")